In [ ]:
import warnings,subprocess, os, pandas as pd, geopandas as gpd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, folium
from IPython.display import display
from run_analysis import buffer_list_mile
from user_inputs import load_selection, update_selection, select_pca_method
crime_col, ctrl_vars = load_selection()   # chosen when run_analysis.py was executed
from common import explore_pca
%matplotlib inline
np.set_printoptions(suppress=True)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
pd.set_option("display.precision", 2)
pd.set_option("display.float_format", "{:.2f}".format)
nb = os.path.basename(__file__) if "__file__" in globals() else "pca_analysis.ipynb"
subprocess.run(["jupyter", "trust", nb])

**Preliminary process**


In [ ]:
gdf = gpd.read_file('../data/final_dataset/extracted_measures.shp')
def generate_street_network_columns(buffer_list_mile):
    # Prefixes for all network metrics
    metrics = ["int_den", "st_den", "nd_deg", "cl_cof", "shtpth", "bt_cnt", "cl_cnt", "pr_cnt"]

    # Same naming rule used in StreetNetworkProcessor
    def bufname(b):
        return str(b).replace(".", "") if (b < 1 and b != 0.1) else str(int(b * 10))

    # Generate all columns
    return [f"{m}{bufname(b)}" for b in buffer_list_mile for m in metrics]
final_list_net = generate_street_network_columns(buffer_list_mile)

**Generates visualization plots for diagnostics or distributions**


In [ ]:
popup_vars = ["id"] + final_list_net + ctrl_vars;gdf_vis = gdf.to_crs(4326)
def fmt(v):
    return f"{v:.2f}" if isinstance(v, (float, int)) and pd.notna(v) else v

features = []
for _, r in gdf_vis.iterrows():
    popup_html = "<br>".join([f"{v}: {fmt(r.get(v, 'NA'))}" for v in popup_vars])
    features.append({
        "type": "Feature",
        "geometry": r.geometry.__geo_interface__,
        "properties": {"id": r["id"], "popup": popup_html}
    })

geojson = {"type": "FeatureCollection", "features": features}

m = folium.Map(
    [gdf_vis.geometry.centroid.y.mean(), gdf_vis.geometry.centroid.x.mean()],
    zoom_start=11, tiles="CartoDB positron"
)

folium.GeoJson(
    geojson,
    tooltip=folium.GeoJsonTooltip(fields=["id"], aliases=["ID:"]),
    popup=folium.GeoJsonPopup(fields=["popup"], labels=False, max_width=350)
).add_to(m)

m

**Exploratory data inspection.**


In [ ]:
print("📊 Descriptive Statistics\n");display(gdf[final_list_net + ctrl_vars].describe().T.round(3))

**Missing values.**

**Note: Any missing values must be handled before PCA analysis.**


In [ ]:
missing_values = gdf[final_list_net + ctrl_vars].isnull().sum();missing_values_filtered = missing_values[missing_values > 0]
if len(missing_values_filtered) == 0:
    print("No missing values in any feature")
else:
    print(missing_values_filtered)
    print("\n \033[1mFor further processing, missing values must be handled!\033[0m")

**Histogram and bar plot for diagnostics or distributions**


In [ ]:
numerical_columns = gdf[final_list_net + ctrl_vars].select_dtypes(include=['number']).columns;print("histogram and bar plots of each variable")
for col in numerical_columns:
    print(col)
    print('Skew :', round(gdf[final_list_net + ctrl_vars][col].skew(), 2))
    plt.figure(figsize = (10, 4))
    plt.subplot(1, 2, 1)
    sns.distplot(gdf[final_list_net + ctrl_vars][col], axlabel=col)
    plt.subplot(1, 2, 2)
    sns.boxplot(x = gdf[final_list_net + ctrl_vars][col])
    plt.show()

**Boxplot for diagnostics or distributions**


In [ ]:
selected_cols = ["id"] + final_list_net + ctrl_vars; sub = gdf[selected_cols]
numerical_columns = sub.select_dtypes(include='number').columns.tolist()

if "id" in numerical_columns:
    numerical_columns.remove("id")

for col in numerical_columns:
    print(f"Boxplot for {col}")

    plt.figure(figsize=(10, 5))

    # Boxplot
    sns.boxplot(data=sub, x=col)

    # Overlaid raw points
    sns.stripplot(data=sub, x=col, color='black', jitter=True, alpha=1)

    plt.title(f"Boxplot for {col}")
    plt.tight_layout()
    plt.show()

**PCA: choose the number of components interactively**

PCA is used to collapse each block of variables (street-network measures, socio-spatial controls) into a few uncorrelated components, so that the regression models in `spatial_modeling.Rmd` are not affected by multicollinearity. The number of components is a judgement call, so each block is handled in three stages:

1. **Overview** (independent of *n*): correlation heatmap of the variables, scree plot of all eigenvalues with the Kaiser line and cumulative variance, eigenvalue table and rule-of-thumb counts.
2. **Inspect candidate solutions**: type a number *n* and the full report for that solution is displayed: unrotated and rotated loadings (tables and heatmaps), communalities, variance explained, salient loadings per component, cross-loading variables, variables captured by no component and low communalities. Repeat with other values of *n* as often as needed.
3. **Final choice**: a prompt asks for the number of components to retain (Enter keeps the last one inspected). The choice is saved to `user_selection.json` for the modeling stage.

Two settings are asked for first. The **extraction method**: the paper describes the procedure as PCA with varimax rotation, while the original computation used `factor_analyzer`'s default (`minres`, minimum-residual common factor analysis); `principal` gives principal components in the strict sense. The **rotation** is set in the same cell (varimax in the paper). The published study retained 4 network and 9 socio-spatial components.

In [ ]:
pca_method   = select_pca_method()   # asks: principal / minres / ml (see note above)
pca_rotation = "varimax"             # rotation of the retained components (paper: varimax)

**Street-network measures: overview, candidate solutions, final choice**

In [ ]:
n_net, pca_net = explore_pca(gdf, final_list_net, label="street network",
                             rotation=pca_rotation, method=pca_method)

**Socio-spatial controls: overview, candidate solutions, final choice**

In [ ]:
n_ctrl, pca_ctrl = explore_pca(gdf, ctrl_vars, label="socio-spatial controls",
                               rotation=pca_rotation, method=pca_method)

# Record the choices next to the variable selection made in run_analysis.py
update_selection(pca_method=pca_method, pca_rotation=pca_rotation,
                 pca_factors_network=n_net, pca_factors_ctrl=n_ctrl)

**Export the rotated component scores for the modeling stage** (`final_components.shp`)

In [ ]:
rotated_net_scores = pca_net["rotated_scores"].copy()
rotated_net_scores.columns = [f"PC{i+1}_net" for i in range(n_net)]
rotated_ctrl_scores = pca_ctrl["rotated_scores"].copy()
rotated_ctrl_scores.columns = [f"PC{i+1}_ctrl" for i in range(n_ctrl)]

final_gdf = pd.concat([gdf[["id", "geometry"]].reset_index(drop=True),
                       rotated_net_scores, rotated_ctrl_scores], axis=1)
final_gdf = gpd.GeoDataFrame(final_gdf, geometry="geometry", crs=gdf.crs)
final_gdf.to_file('../data/final_dataset/final_components.shp')
print(f"Exported {len(final_gdf)} rows: {n_net} network + {n_ctrl} control components.")